In [0]:
from pyspark.sql import functions as F

bronze_match_details = spark.read.table("opendota.bronze.match_details")

spark.sql("CREATE SCHEMA IF NOT EXISTS opendota.silver")

silver_match_details = (
    bronze_match_details
    .drop(
        "__index_level_0__", 
        "_airbyte_meta", 
        "_airbyte_raw_id", 
        "_airbyte_extracted_at", 
        "_airbyte_generation_id", 
        "_ab_source_file_url", 
        "_ab_source_file_last_modified"
    )
)

silver_match_details = (
    silver_match_details
    .withColumnRenamed("leagueid", "league_id")
    .withColumn("start_time", F.from_unixtime(F.col("start_time")).cast("timestamp"))
    .withColumn("match_date", F.to_date("start_time"))
    .withColumn("duration", F.col("duration").cast("int"))
    .withColumn("pre_game_duration", F.col("pre_game_duration").cast("int"))
    .withColumn("first_blood_time", F.col("first_blood_time").cast("int"))
    .withColumn("radiant_score", F.col("radiant_score").cast("int"))
    .withColumn("dire_score", F.col("dire_score").cast("int"))
    .withColumn("human_players", F.col("human_players").cast("int"))
    .withColumn("radiant_name", F.trim(F.col("radiant_name")))
    .withColumn("dire_name", F.trim(F.col("dire_name")))
    .withColumn("radiant_logo", F.trim(F.col("radiant_logo")))
    .withColumn("dire_logo", F.trim(F.col("dire_logo")))
    .withColumn("replay_url", F.trim(F.col("replay_url")))
    .withColumn(
        "radiant_team_complete",
        F.when(F.col("radiant_team_complete") == 1, True)
         .when(F.col("radiant_team_complete") == 0, False)
    )
    .withColumn(
        "dire_team_complete",
        F.when(F.col("dire_team_complete") == 1, True)
         .when(F.col("dire_team_complete") == 0, False)
    )
    .withColumn("processed_at", F.current_timestamp())
)

(
    silver_match_details.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("opendota.silver.match_details")
)